# Consolidated Random Forest Classifiers

Train and evaluate random forest classifiers for three modalities:
- **Morphology (M)**: dendrite morphological features
- **Electrophysiology (E)**: patch-clamp features with paradigm variants
- **Morphoelectric (ME)**: combined M+E features

Generates confusion matrices and saves results for downstream analysis.

## Workflow
1. Load shared data (metadata, colors, label ordering)
2. For each modality:
   - Load features
   - Prepare metadata
   - Run classifier variants
   - Save PDFs and structured outputs
3. Validate results against legacy notebooks

In [1]:
# Imports
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

# Import classifier utilities
import sys
sys.path.append('/root/capsule/code/figures')
from classifier_utils import (
    get_modality_configs,
    load_modality_features,
    prepare_metadata,
    run_standard_variant,
    run_acute_only_variant,
    run_acute_train_culture_test_variant,
    run_balance_paradigm_variant
)
from core_utils import get_seaad_colors, get_tx_order

warnings.filterwarnings('ignore', category=FutureWarning)

## Configuration and Shared Data Loading

In [2]:
# Paths
DATA_ROOT = Path('/root/capsule/data')
SCRATCH_ROOT = Path('/root/capsule/scratch/20260722')
metadata_path = DATA_ROOT / 'human' / 'metadata' / '20260722' / 'human_metadata.csv'
ttype_layer_file = DATA_ROOT / 'formatting_dicts' / 'ttype_layer_dict.json'
l23_ttype_rename_file = DATA_ROOT / 'formatting_dicts' / 'L23_ttype_rename.json'

# Load formatting dictionaries
with open(ttype_layer_file, "r") as f:
    ttype_to_layer_dict = json.load(f)
with open(l23_ttype_rename_file, "r") as f:
    L23_ttype_rename_dict = json.load(f)

# Get color dictionary
color_dict = get_seaad_colors()

# Get transcriptomic ordering
ttype_order, subclass_order = get_tx_order()
layer_subclass_order = ['L2 IT'] + [l.replace('L2/3 IT', 'L3 IT') for l in subclass_order]

# Construct label order dictionary
label_order = {
    'subclass_label': subclass_order,
    'layer_subclass': layer_subclass_order,
    't_type': ttype_order,
    'layer': ['Layer 2', 'Layer 3', 'Layer 4', 'Layer 5', 'Layer 6']
}

# Get modality configurations
MODALITY_CONFIGS = get_modality_configs(DATA_ROOT)

print("✓ Configuration and shared data loaded")

✓ Configuration and shared data loaded


## Main Execution Loop

Run classifiers for each modality, depth type, and variant.

In [3]:
# Configuration: which modalities and label columns to run
modalities_to_run = ['morphology', 'morphoelectric', 'electrophysiology']  # All three modalities
label_columns_to_run = ['subclass_label']  # Can expand to other label types
depth_types = ['withDepth', 'withoutDepth']

print("Starting classifier training...\n")

for modality_name in modalities_to_run:
    config = MODALITY_CONFIGS[modality_name]
    print(f"\n{'#'*80}")
    print(f"# MODALITY: {config['name']}")
    print(f"{'#'*80}\n")
    
    # Load features
    features = load_modality_features(config, modality_name)
    
    # Prepare metadata
    metadata = prepare_metadata(
        metadata_path,
        features.index,
        ttype_to_layer_dict,
        L23_ttype_rename_dict,
        include_paradigm=config['has_paradigm']
    )
    
    # Remove von Economo neuron from features
    ven_spid = 1263702688
    features = features.drop(index=ven_spid, errors='ignore')
    
    # Create output directories
    scratch_dir = SCRATCH_ROOT / config['output_dir']
    os.makedirs(scratch_dir, exist_ok=True)
    
    structured_output_dir = DATA_ROOT / 'human'
    os.makedirs(structured_output_dir / 'classifiers' / '20260722' / config['short_name'].lower(), exist_ok=True)
    
    # Save ME cell list if requested
    if config.get('save_cell_list', False):
        me_cell_list_path = scratch_dir / 'ME_classifier_cells.csv'
        metadata.reset_index().to_csv(me_cell_list_path, index=False)
        print(f"Saved ME cell list to {me_cell_list_path}")
    
    # Loop through output variants
    for output_variant in config['output_variants']:
        
        if output_variant == 'standard':
            # Standard variant: loop through stratification types and depths
            for stratify_type in config['stratify_variants']:
                for depth_type in depth_types:
                    
                    # Construct PDF path
                    if modality_name == 'electrophysiology':
                        pdf_basename = config['pdf_basename_standard'].format(
                            min_cells=config['min_cells'],
                            depth_type=depth_type,
                            stratify=stratify_type
                        )
                    else:
                        pdf_basename = config['pdf_basename'].format(
                            min_cells=config['min_cells'],
                            depth_type=depth_type
                        )
                    
                    pdf_path = scratch_dir / pdf_basename
                    
                    with PdfPages(pdf_path) as pdf:
                        for label_column in label_columns_to_run:
                            run_standard_variant(
                                features, metadata, config,
                                label_column, depth_type, stratify_type,
                                label_order, color_dict,
                                structured_output_dir, pdf
                            )
                    
                    print(f"✓ Saved PDF: {pdf_path}")
        
        elif output_variant == 'acuteOnly':
            # Acute-only variant
            for depth_type in depth_types:
                pdf_basename = config['pdf_basename_acute_only'].format(
                    min_cells=config['min_cells'],
                    depth_type=depth_type
                )
                pdf_path = scratch_dir / pdf_basename
                
                with PdfPages(pdf_path) as pdf:
                    for label_column in label_columns_to_run:
                        run_acute_only_variant(
                            features, metadata, config,
                            label_column, depth_type,
                            label_order, color_dict,
                            structured_output_dir, pdf
                        )
                
                print(f"✓ Saved PDF: {pdf_path}")
        
        elif output_variant == 'acuteTrain_cultureTest':
            # Train on acute, test on culture
            for depth_type in depth_types:
                pdf_basename = config['pdf_basename_acute_train_culture'].format(
                    min_cells=config['min_cells'],
                    depth_type=depth_type
                )
                pdf_path = scratch_dir / pdf_basename
                
                with PdfPages(pdf_path) as pdf:
                    for label_column in label_columns_to_run:
                        run_acute_train_culture_test_variant(
                            features, metadata, config,
                            label_column, depth_type,
                            label_order, color_dict,
                            structured_output_dir, pdf
                        )
                
                print(f"✓ Saved PDF: {pdf_path}")
        
        elif output_variant == 'balanceParadigm':
            # Balance acute/culture in train and test
            for depth_type in depth_types:
                pdf_basename = config['pdf_basename_balance'].format(
                    min_cells=config['min_cells'],
                    depth_type=depth_type
                )
                pdf_path = scratch_dir / pdf_basename
                
                with PdfPages(pdf_path) as pdf:
                    for label_column in label_columns_to_run:
                        run_balance_paradigm_variant(
                            features, metadata, config,
                            label_column, depth_type,
                            label_order, color_dict,
                            structured_output_dir, pdf
                        )
                
                print(f"✓ Saved PDF: {pdf_path}")

print("\n" + "="*80)
print("All classifiers completed!")
print("="*80)

Starting classifier training...


################################################################################
# MODALITY: Morphology
################################################################################

Loaded 327 Morphology cells
Prepared metadata for 326 cells

STANDARD VARIANT: Morphology
Label: subclass_label, Depth: withDepth, Stratify: simple
Label column: subclass_label
Labels meeting threshold (>=4 cells): 9
Total cells after filtering: 321
Train/test split: 256/65 (80/20)
Stratification mode: simple
Best RF params: {'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}
CV (3 folds) | Accuracy: 0.797 ± 0.028 | F1: 0.793 ± 0.026
Test set | Accuracy: 0.846 | F1: 0.836
Saved results to /root/capsule/data/human/classifiers/20260722/m with prefix morphology_subclass_withDepth_simple_standard
✓ Saved PDF: /root/capsule/scratch/20260722/M_classifier/human_excitatory_morpho_classifier_rf_min_4_cells_rawFeatures_withDepth.pdf

STANDARD VA